In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils import class_weight
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
import h5py
import keras

2026-07-22 14:46:09.076807: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-22 14:46:09.101325: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-22 14:46:09.123604: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-22 14:46:09.130243: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-22 14:46:09.148339: I tensorflow/core/platform/cpu_feature_guar

In [ ]:
# Load your Data

In [4]:
# ==================== Token 级专家 (FFN) ====================
class Expert(keras.layers.Layer):
    """单个专家：SwiGLU 门控 FFN"""
    def __init__(self, embed_dim, intermediate_dim, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.embed_dim = embed_dim
        self.intermediate_dim = intermediate_dim

        self.gate_proj = keras.layers.Dense(
            self.intermediate_dim, use_bias=False,
            kernel_initializer="glorot_uniform", name="gate_proj"
        )
        self.up_proj = keras.layers.Dense(
            self.intermediate_dim, use_bias=False,
            kernel_initializer="glorot_uniform", name="up_proj"
        )
        self.down_proj = keras.layers.Dense(
            self.embed_dim, use_bias=False,
            kernel_initializer="glorot_uniform", name="down_proj"
        )
        self.act = keras.layers.Activation("silu")

    def call(self, x):
        gate = self.act(self.gate_proj(x))
        up = self.up_proj(x)
        return self.down_proj(gate * up)

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "intermediate_dim": self.intermediate_dim,
        })
        return config

In [5]:
# ==================== Token 级稀疏 MoE 层（修复版）====================
class SparseMoELayer(keras.layers.Layer):
    def __init__(
        self,
        embed_dim,
        intermediate_dim,
        num_experts=5,
        num_routed_experts=4,
        top_k=2,
        load_balance_coef=0.01,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.intermediate_dim = intermediate_dim
        self.num_experts = num_experts
        self.num_routed_experts = num_routed_experts
        self.top_k = top_k
        self.load_balance_coef = load_balance_coef
        assert num_experts == num_routed_experts + 1

        self.experts = [
            Expert(self.embed_dim, self.intermediate_dim, name=f"expert_{i}")
            for i in range(self.num_experts)
        ]
        self.router = keras.layers.Dense(
            self.num_routed_experts,
            use_bias=False,
            kernel_initializer="glorot_uniform",
            name="token_router"
        )

    def call(self, x, training=False):
        batch_size = keras.ops.shape(x)[0]
        seq_len = keras.ops.shape(x)[1]
        x_flat = keras.ops.reshape(x, (-1, self.embed_dim))

        # 共享专家
        shared_output = self.experts[0](x_flat)

        # 路由
        router_logits = self.router(x_flat)
        router_probs = keras.ops.softmax(router_logits, axis=-1)
        top_k_probs, top_k_indices = keras.ops.top_k(router_probs, k=self.top_k)
        top_k_probs = top_k_probs / (
            keras.ops.sum(top_k_probs, axis=-1, keepdims=True) + 1e-9
        )

        # 负载均衡损失
        if training:
            aux_loss = self._compute_load_balance_loss(router_probs, top_k_indices)
            self.add_loss(self.load_balance_coef * aux_loss)

        # 稀疏专家聚合 —— 纯张量操作，无 Python 条件分支
        combined_output = shared_output
        for k in range(self.top_k):
            expert_weights = top_k_probs[:, k]      # (num_tokens,)
            expert_indices = top_k_indices[:, k]    # (num_tokens,)

            for expert_id in range(self.num_routed_experts):
                mask = keras.ops.equal(expert_indices, expert_id)  # (num_tokens,) bool tensor

                # 修复：不用 if ops.any(mask)，改用 ops.where
                # 将 mask 转换为与 x_flat 同 dtype 的权重 (0 或 1)
                mask_float = keras.ops.cast(mask, x_flat.dtype)  # (num_tokens,)
                mask_float = keras.ops.expand_dims(mask_float, axis=-1)  # (num_tokens, 1)

                # 构造输入：选中位置用原值，未选中位置用 0
                expert_input = x_flat * mask_float  # (num_tokens, embed_dim)

                # 通过专家
                expert_out = self.experts[expert_id + 1](expert_input)

                # 加权：只保留选中位置的输出
                weight = keras.ops.expand_dims(expert_weights, axis=-1)  # (num_tokens, 1)
                weighted_out = expert_out * weight * mask_float

                combined_output = combined_output + weighted_out

        output = keras.ops.reshape(combined_output, (batch_size, seq_len, self.embed_dim))
        return output

    def _compute_load_balance_loss(self, router_probs, top_k_indices):
        num_tokens = keras.ops.cast(keras.ops.shape(router_probs)[0], "float32")
        num_routed = self.num_routed_experts
        expert_mask = keras.ops.one_hot(top_k_indices, num_routed)
        expert_mask = keras.ops.sum(expert_mask, axis=1)
        f = keras.ops.sum(expert_mask, axis=0) / num_tokens
        P = keras.ops.mean(router_probs, axis=0)
        return num_routed * keras.ops.sum(f * P)

    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "intermediate_dim": self.intermediate_dim,
            "num_experts": self.num_experts,
            "num_routed_experts": self.num_routed_experts,
            "top_k": self.top_k,
            "load_balance_coef": self.load_balance_coef,
        })
        return config

In [6]:
# ==================== 句子级 MoE 层（硬路由 Top-1，修复版）====================
class SentenceLevelMoE(keras.layers.Layer):
    def __init__(
        self,
        embed_dim,
        intermediate_dim,
        num_sentence_experts=3,
        num_token_experts=5,
        num_routed_token_experts=4,
        token_top_k=2,
        token_load_balance_coef=0.01,
        sentence_router_coef=0.1,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.intermediate_dim = intermediate_dim
        self.num_sentence_experts = num_sentence_experts
        self.num_token_experts = num_token_experts
        self.num_routed_token_experts = num_routed_token_experts
        self.token_top_k = token_top_k
        self.token_load_balance_coef = token_load_balance_coef
        self.sentence_router_coef = sentence_router_coef

        self.sentence_proj = keras.layers.Dense(
            self.embed_dim,
            activation="tanh",
            use_bias=False,
            name="sentence_proj"
        )
        self.sentence_router = keras.layers.Dense(
            self.num_sentence_experts,
            use_bias=False,
            kernel_initializer="glorot_uniform",
            name="sentence_router"
        )
        self.sentence_experts = [
            SparseMoELayer(
                embed_dim=self.embed_dim,
                intermediate_dim=self.intermediate_dim,
                num_experts=self.num_token_experts,
                num_routed_experts=self.num_routed_token_experts,
                top_k=self.token_top_k,
                load_balance_coef=self.token_load_balance_coef,
                name=f"sentence_expert_{i}"
            )
            for i in range(self.num_sentence_experts)
        ]

    def call(self, x, sentence_labels=None, training=False):
        batch_size = keras.ops.shape(x)[0]

        # 句子级表示
        sentence_repr = keras.ops.mean(x, axis=1)
        sentence_repr = self.sentence_proj(sentence_repr)
        sentence_logits = self.sentence_router(sentence_repr)

        # 有监督路由损失
        if training and sentence_labels is not None:
            # 确保 sentence_labels 是 1D
            sentence_labels = keras.ops.reshape(keras.ops.cast(sentence_labels, "int32"),[-1])
            router_loss = keras.ops.sparse_categorical_crossentropy(
                sentence_labels, sentence_logits, from_logits=True
            )
            router_loss = keras.ops.mean(router_loss)
            self.add_loss(self.sentence_router_coef * router_loss)

        # 硬路由决策
        if training and sentence_labels is not None:
            selected_expert_ids = keras.ops.reshape(keras.ops.cast(sentence_labels, "int32"),[-1])
        else:
            selected_expert_ids = keras.ops.argmax(sentence_logits, axis=-1)

        # ========== 修复：纯张量操作，无 Python 条件分支 ==========
        output = keras.ops.zeros_like(x)  # (batch, seq, embed)

        for expert_id in range(self.num_sentence_experts):
            # mask: (batch,) bool
            mask = keras.ops.equal(selected_expert_ids, expert_id)
            # 扩展为 (batch, 1, 1) 用于广播
            mask_3d = keras.ops.expand_dims(keras.ops.expand_dims(mask, axis=-1), axis=-1)
            # 转 dtype
            mask_float = keras.ops.cast(mask_3d, x.dtype)

            # 所有样本都过这个专家，但未选中的位置会被 mask 为 0
            expert_out = self.sentence_experts[expert_id](x, training=training)

            # 只保留选中该专家的输出
            masked_out = expert_out * mask_float

            # 累加
            output = output + masked_out

        return output

    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "intermediate_dim": self.intermediate_dim,
            "num_sentence_experts": self.num_sentence_experts,
            "num_token_experts": self.num_token_experts,
            "num_routed_token_experts": self.num_routed_token_experts,
            "token_top_k": self.token_top_k,
            "token_load_balance_coef": self.token_load_balance_coef,
            "sentence_router_coef": self.sentence_router_coef,
        })
        return config


In [7]:
# ==================== 双层 MoE Transformer 块 ====================
class SentenceMoETransformerBlock(keras.layers.Layer):
    def __init__(
        self,
        embed_dim,
        head_dim,
        num_query_heads,
        num_key_value_heads,
        intermediate_dim,
        num_sentence_experts=3,
        num_token_experts=5,
        num_routed_token_experts=4,
        token_top_k=2,
        token_load_balance_coef=0.01,
        sentence_router_coef=0.1,
        dropout_rate=0.1,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.head_dim = head_dim
        self.num_query_heads = num_query_heads
        self.num_key_value_heads = num_key_value_heads
        self.intermediate_dim = intermediate_dim
        self.num_sentence_experts = num_sentence_experts
        self.num_token_experts = num_token_experts
        self.num_routed_token_experts = num_routed_token_experts
        self.token_top_k = token_top_k
        self.token_load_balance_coef = token_load_balance_coef
        self.sentence_router_coef = sentence_router_coef
        self.dropout_rate = dropout_rate

        self.norm1 = keras.layers.LayerNormalization(epsilon=1e-6, name="pre_attn_norm")
        self.norm2 = keras.layers.LayerNormalization(epsilon=1e-6, name="pre_moe_norm")

        self.gqa = keras.layers.GroupQueryAttention(
            head_dim=self.head_dim,
            num_query_heads=self.num_query_heads,
            num_key_value_heads=self.num_key_value_heads,
            dropout=self.dropout_rate,
            use_bias=False,
            name="gqa_attention"
        )

        self.sentence_moe = SentenceLevelMoE(
            embed_dim=self.embed_dim,
            intermediate_dim=self.intermediate_dim,
            num_sentence_experts=self.num_sentence_experts,
            num_token_experts=self.num_token_experts,
            num_routed_token_experts=self.num_routed_token_experts,
            token_top_k=self.token_top_k,
            token_load_balance_coef=self.token_load_balance_coef,
            sentence_router_coef=self.sentence_router_coef,
            name="sentence_moe"
        )

        self.dropout1 = keras.layers.Dropout(self.dropout_rate)
        self.dropout2 = keras.layers.Dropout(self.dropout_rate)

    def call(self, x, sentence_labels=None, training=False, attention_mask=None):
        # 注意力子层
        residual = x
        x = self.norm1(x)
        attn_output = self.gqa(
            query=x, value=x, key=x,
            attention_mask=attention_mask,
            training=training,
            use_causal_mask=True
        )
        attn_output = self.dropout1(attn_output, training=training)
        x = residual + attn_output

        # 句子级 MoE 子层（硬路由）
        residual = x
        x = self.norm2(x)
        moe_output = self.sentence_moe(
            x,
            sentence_labels=sentence_labels,
            training=training
        )
        moe_output = self.dropout2(moe_output, training=training)
        x = residual + moe_output

        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "head_dim": self.head_dim,
            "num_query_heads": self.num_query_heads,
            "num_key_value_heads": self.num_key_value_heads,
            "intermediate_dim": self.intermediate_dim,
            "num_sentence_experts": self.num_sentence_experts,
            "num_token_experts": self.num_token_experts,
            "num_routed_token_experts": self.num_routed_token_experts,
            "token_top_k": self.token_top_k,
            "token_load_balance_coef": self.token_load_balance_coef,
            "sentence_router_coef": self.sentence_router_coef,
            "dropout_rate": self.dropout_rate,
        })
        return config
    
    def compute_output_shape(self, input_shape):
        # 输入输出同形: (batch, seq, embed_dim) -> (batch, seq, embed_dim)
        return input_shape

In [ ]:
# ==================== Total model ====================
def ETMOEformer(embed_dim=64, intermediate_dim=168, cls=2):
    inputs = keras.Input(shape=(1250, 1), name="input_series")
    sentence_labels = keras.Input(shape=(1,), dtype="int32", name="sentence_labels")

    # ================================ Embedding + 位置编码 ================================
    embedcnn1 = keras.layers.Conv1D(
        8, kernel_size=5, padding="causal", activation="relu", name="embed_cnn1"
    )(inputs)
    embedcnn2 = keras.layers.Conv1D(
        16, kernel_size=5, strides=5, padding="causal", activation="relu", name="embed_cnn2"
    )(embedcnn1)
    embedcnn3 = keras.layers.Conv1D(
        embed_dim, kernel_size=5, dilation_rate=2, padding="causal", activation="relu", name="embed_cnn3"
    )(embedcnn2)

    pos = keras.layers.Dropout(0.05,name="pos")(embedcnn3)
    
    # ================================ GQA 子层 ================================
    smt1 = SentenceMoETransformerBlock(
        embed_dim=embed_dim,
        head_dim=16,
        num_query_heads=4,
        num_key_value_heads=2,
        intermediate_dim=intermediate_dim,
        name=f"smt_block_{1}"
    )(pos, sentence_labels=sentence_labels)

    smt2 = SentenceMoETransformerBlock(
        embed_dim=embed_dim,
        head_dim=16,
        num_query_heads=4,
        num_key_value_heads=2,
        intermediate_dim=intermediate_dim,
        name=f"smt_block_{2}"
    )(smt1, sentence_labels=sentence_labels)

    smt3 = SentenceMoETransformerBlock(
        embed_dim=embed_dim,
        head_dim=16,
        num_query_heads=4,
        num_key_value_heads=2,
        intermediate_dim=intermediate_dim,
        name=f"smt_block_{3}"
    )(smt2, sentence_labels=sentence_labels)

    # ================================= Classification ==================================
    clsgap = keras.layers.GlobalAvgPool1D(name='clsgap')(smt3)
    output = keras.layers.Dense(units=cls,activation='softmax',name='output')(clsgap)

    model = keras.Model(inputs={'input_series':inputs, 'sentence_labels':sentence_labels}, outputs={'output':output}, name="ET-MOE-former")
    return model

In [ ]:
model = ETMOEformer()
model.summary()

2026-07-22 14:46:12.165412: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38104 MB memory:  -> device: 0, name: NVIDIA A100-PCIE-40GB, pci bus id: 0000:19:00.0, compute capability: 8.0


Model: "ET-MOE-former"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_series        │ (None, 1250, 1)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed_cnn1 (Conv1D) │ (None, 1250, 8)   │         48 │ input_series[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed_cnn2 (Conv1D) │ (None, 250, 16)   │        656 │ embed_cnn1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed_cnn3 (Conv1D) │ (None, 250, 64)   │      5,184 │ embed_cnn2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pos (Dropout)       │ (None, 250, 64)   │          0 │ embed_cnn3[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sentence_labels     │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ smt_block_1         │ (None, 250, 64)   │    501,440 │ pos[0][0],        │
│ (SentenceMoETransf… │                   │            │ sentence_labels[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ smt_block_2         │ (None, 250, 64)   │    501,440 │ smt_block_1[0][0… │
│ (SentenceMoETransf… │                   │            │ sentence_labels[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ smt_block_3         │ (None, 250, 64)   │    501,440 │ smt_block_2[0][0… │
│ (SentenceMoETransf… │                   │            │ sentence_labels[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ clsgap              │ (None, 64)        │          0 │ smt_block_3[0][0] │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 2)         │        130 │ clsgap[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,510,338 (5.76 MB)

 Trainable params: 1,510,338 (5.76 MB)

 Non-trainable params: 0 (0.00 B)

# Training testing

In [ ]:
# ============================ V2 ================================================
keras.utils.set_random_seed(42)

epc = 60
bs = 128
lr = 2.5e-4


for sub in np.arange(start=0,stop=23,step=1) :
    
    train_lab = lab_seg[lab_mtx_5s[:,-1]!=sub,1]
    test_lab = lab_seg[lab_mtx_5s[:,-1]==sub,1]

    train_onehot_lab = keras.utils.to_categorical(train_lab, num_classes=2)
    test_onehot_lab = keras.utils.to_categorical(test_lab, num_classes=2)

    spw = class_weight.compute_sample_weight(class_weight='balanced', y=train_lab)
    spw_alpha = class_weight.compute_class_weight(class_weight='balanced', classes=np.array([0,1]), y=train_lab)

    train_scent = lab_mtx_5s[lab_mtx_5s[:,-1]!=sub,-3]
    test_scent = lab_mtx_5s[lab_mtx_5s[:,-1]==sub,-3]
    
    train_sig = ppg_sig_5s[lab_mtx_5s[:,-1]!=sub,::,:]
    test_sig = ppg_sig_5s[lab_mtx_5s[:,-1]==sub,::,:]

    ptp_emocls = ETMOEformer(cls=2)
    
    opt_ptp = keras.optimizers.AdamW(learning_rate=lr, global_clipnorm=1.)
    ptp_emocls.compile(loss={'output':keras.losses.CategoricalFocalCrossentropy(alpha=spw_alpha)},
                       loss_weights={'output':10},
                       optimizer=opt_ptp,
                       metrics={'output':["categorical_accuracy",keras.metrics.F1Score(average="weighted"),keras.metrics.F1Score(average="macro")]})
    ckpt_ptp_fp = f"deap/avg_v2_a2/cls_loso_v2_s{sub}.valbest.keras"
    ckpt_ptp = keras.callbacks.ModelCheckpoint(ckpt_ptp_fp, monitor='val_f1_score_1', verbose=0, save_best_only=True, mode='max')
    _ = ptp_emocls.fit(x={"input_series": train_sig[:,:,:],"sentence_labels": train_scent}, 
                       y={'output':train_onehot_lab},
                       batch_size=bs, epochs=epc, verbose=0, sample_weight=spw, 
                       validation_data=({"input_series": test_sig[:,:,:],"sentence_labels": test_scent},{'output':test_onehot_lab}), 
                       callbacks=[ckpt_ptp])
    ptp_emocls.load_weights(ckpt_ptp_fp)
    _ = ptp_emocls.evaluate(x={"input_series": test_sig[:,:,:],"sentence_labels": test_scent}, y={'output':test_onehot_lab},
                            batch_size=bs, verbose=2)
    keras.utils.clear_session(free_memory=True)

# Confusion Metrics

In [ ]:
plt.rcParams.update({'font.size': 24})

yp1_tt = np.array([])
yt1_tt = np.array([])

ptp_emocls = ETMOEformer(cls=2)

for sub in np.arange(23) :
    # print(f'Testing for Subject {sub+1} ...',flush=True)
    test_lab = lab_seg[lab_mtx_5s[:,-1]==sub,1]
    test_onehot_lab = keras.utils.to_categorical(test_lab, num_classes=2)
    yt1_tt = np.append(yt1_tt,test_lab)
    
    test_sig = ppg_sig_5s[lab_mtx_5s[:,-1]==sub,::,:]
    
    ckpt_ptp_fp = f"deap/avg_v2_a2/cls_loso_v2_s{sub}.valbest.keras"
    ptp_emocls.load_weights(ckpt_ptp_fp)
    y_pred = ptp_emocls.predict(x={"input_series": test_sig[:,:,:],"sentence_labels": test_scent},batch_size=128,verbose=0)['output']
    y_hat = y_pred.argmax(axis=1)
    yp1_tt = np.append(yp1_tt,y_hat)

cm1 = confusion_matrix(yt1_tt, yp1_tt,normalize='true')
cm_display1 = ConfusionMatrixDisplay(cm1,display_labels=['Negative','Positive']).plot(cmap='Blues',colorbar=False)